# event camera scratch space

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import numpy as np
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import finufft
from scipy.stats import chi, chi2
from tqdm import tqdm
import os
from sptflux import flux_probing, ihpp_data, plotting, config, utils

# note: the event camera output for the website was subsampled from 5000 frames -> 200 frames

In [ ]:
def make_moving_gaussian(amplitude, sigma, velocity, period):
    """
    Returns a function lambda(x, y, t) representing a Gaussian ball 
    moving back and forth along the x-axis.
    
    Args:
        amplitude (float): Peak intensity.
        sigma (float): Standard deviation (spread) of the Gaussian.
        velocity (float): (Unused in simple oscillator, derived from period/bounds) 
                          controls speed if we weren't fixed to [0,1].
                          Here, we use 'period' to dictate the full cycle duration.
        period (float): Time to complete one full back-and-forth cycle.
        
    Returns:
        function: lambda(x, y, t) -> intensity
    """
    def intensity_func(x, y, t):
        # Center x oscillation: moves between 0.2 and 0.8 to stay within [0,1] mostly
        # x_c(t) = center + oscillating_offset
        omega = 2 * np.pi / period
        # Oscillation centered at 0.5, amplitude 0.3 (range 0.2 to 0.8)
        x_c = 0.5 + 0.3 * np.sin(omega * t)
        y_c = 0.5 # Fixed y center
        
        # Gaussian formula
        exponent = -((x - x_c)**2 + (y - y_c)**2) / (2 * sigma**2)
        return amplitude * np.exp(exponent)
        
    return intensity_func

def make_flashing_gaussian(amplitude, sigma, period, fluc_intensity=0.5):
    """
    Returns a function lambda(x, y, t) representing a stationary Gaussian ball 
    that flashes (intensity modulates sinusoidally).
    
    Args:
        amplitude (float): Peak intensity.
        sigma (float): Standard deviation.
        period (float): Time for one full flash cycle.
        
    Returns:
        function: lambda(x, y, t) -> intensity
    """
    def intensity_func(x, y, t):
        x_c, y_c = 0.5, 0.5
        omega = 2 * np.pi / period
        
        # Modulation: oscillates between 0 and 1
        # (sin goes -1 to 1, +1 makes it 0 to 2, /2 makes it 0 to 1)
        modulation = (fluc_intensity * np.sin(omega * t)) + 0.5
        
        exponent = -((x - x_c)**2 + (y - y_c)**2) / (2 * sigma**2)
        return amplitude * modulation * np.exp(exponent)
        
    return intensity_func

def make_moving_circle(amplitude, radius, period):
    """
    Returns a function lambda(x, y, t) representing a solid circle (step function)
    moving back and forth along the x-axis.
    
    Args:
        amplitude (float): Intensity inside the circle.
        radius (float): Radius of the circle (0 to 0.5 typically).
        period (float): Time to complete one full back-and-forth cycle.
        
    Returns:
        function: lambda(x, y, t) -> intensity
    """
    def intensity_func(x, y, t):
        # Same oscillatory motion: Center moves between 0.2 and 0.8
        omega = 2 * np.pi / period
        x_c = 0.5 + 0.3 * np.cos(omega * t)
        y_c = 0.5
        
        # Calculate squared distance from the center
        dist_sq = (x - x_c)**2 + (y - y_c)**2
        
        # Binary selection: Amplitude if inside radius, 0 if outside
        # We use np.where to handle the vector inputs (meshgrids) correctly
        return np.where(dist_sq <= radius**2, amplitude, 0.0)
        
    return intensity_func

def make_flashing_circle(amplitude, radius, period):
    """
    Returns a function lambda(x, y, t) representing a solid circle (step function)
    that flashes (intensity modulates sinusoidally).
    
    Args:
        amplitude (float): Peak intensity inside the circle.
        radius (float): Radius of the circle (0 to 0.5 typically).
        period (float): Time for one full flash cycle.
        
    Returns:
        function: lambda(x, y, t) -> intensity
    """
    def intensity_func(x, y, t):
        x_c, y_c = 0.5, 0.5
        omega = 2 * np.pi / period
        
        # Modulation: oscillates between 0 and 1
        modulation = 0.5 * (1 + np.sin(omega * t))
        
        # Calculate squared distance from the center
        dist_sq = (x - x_c)**2 + (y - y_c)**2
        
        # Binary selection with modulation: amplitude * modulation if inside radius, 0 if outside
        return np.where(dist_sq <= radius**2, amplitude * modulation, 0.0)
        
    return intensity_func

def make_moving_flashing_circle(amplitude, radius, period_flash, period_move):
    """
    Returns a function lambda(x, y, t) representing a solid circle (step function)
    that moves back and forth and flashes.
    
    Args:
        amplitude (float): Peak intensity inside the circle.
        radius (float): Radius of the circle (0 to 0.5 typically).
        period_flash (float): Time for one full flash cycle.
        period_move (float): Time to complete one full back-and-forth movement cycle.
        
    Returns:
        function: lambda(x, y, t) -> intensity
    """
    def intensity_func(x, y, t):
        # Moving center
        omega_move = 2 * np.pi / period_move
        x_c = 0.5 + 0.3 * np.cos(omega_move * t)
        y_c = 0.5
        
        # Flashing modulation
        omega_flash = 2 * np.pi / period_flash
        modulation = 0.5 * (1 + np.sin(omega_flash * t))
        
        # Calculate squared distance from the center
        dist_sq = (x - x_c)**2 + (y - y_c)**2
        
        # Binary selection with modulation: amplitude * modulation if inside radius, 0 if outside
        return np.where(dist_sq <= radius**2, amplitude * modulation, 0.0)
        
    return intensity_func

def sample_lambda(func, T_steps, H, W):
    """Samples the continuous function onto a discrete grid."""
    # Define coordinate grids
    ts = np.linspace(0, 1, T_steps)
    ys = np.linspace(0, 1, H)
    xs = np.linspace(0, 1, W)
    
    # Create meshgrid for vectorization (broadcasting is cleaner/faster)
    # Using 'indexing="ij"' so the order remains (T, Y, X)
    tt, yy, xx = np.meshgrid(ts, ys, xs, indexing='ij')
    
    return func(xx, yy, tt)


from sptflux import ihpp_data
# --- Configuration ---
H, W = 180, 240       # Spatial Resolution
T_steps = 10000       # Temporal Resolution (frames over 1 second)
AMPLITUDE = 100.0    # Intensity peak
SIGMA = 0.1         # Gaussian spread
PERIOD_MOVE = 0.1        # seconds per cycle
PERIOD_FLASH = 0.3     # seconds per cycle
RADIUS = 0.1        # Size of the circle

# # --- 1. Moving Ball Simulation ---
# print("Simulating Moving Ball...")
# moving_lambda = make_moving_gaussian(AMPLITUDE, SIGMA, velocity=None, period=PERIOD)
# video_moving = sample_lambda(moving_lambda, T_steps, H, W)

# # --- 2. Flashing Ball Simulation ---
# print("\nSimulating Flashing Ball...")
# flashing_lambda = make_flashing_gaussian(AMPLITUDE, SIGMA, period=PERIOD)
# video_flashing = sample_lambda(flashing_lambda, T_steps, H, W)

# # 1. Create the function
# print("Simulating Moving Solid Circle...")
# circle_lambda = make_moving_circle(AMPLITUDE, RADIUS, PERIOD)
# circle_lam_flash = make_flashing_circle(AMPLITUDE, RADIUS, PERIOD)

# # 2. Sample to video grid (using the helper from the previous code)
# video_circle = sample_lambda(circle_lambda, T_steps=T_steps, H=H, W=W)
# video_circle_flash = sample_lambda(circle_lam_flash, T_steps=T_steps, H=H, W=W)

dataname = "moving_flashing_circle"
fluxfunc = make_moving_flashing_circle(AMPLITUDE, RADIUS, PERIOD_MOVE, PERIOD_FLASH)
video_fluxfunc = sample_lambda(fluxfunc, T_steps=T_steps, H=H, W=W)
video_fluxfunc += 10

texp = 10.0
ihpp_data.to_video(video_fluxfunc, f"data/event/{dataname}.mp4", cmap="gray", playback_fps=30, vmin=0)
paths = ihpp_data.to_video(video_fluxfunc, f"data/event/{dataname}", cmap="gray", vmin=0)
with open(f"data/event/{dataname}/info.txt", "w") as f:
    for i, p in enumerate(paths):
        time_us = int(texp * i / T_steps * 1e6)
        f.write(f"./data_samples/interp/{dataname}/{p.name} {str(time_us).zfill(12)}\n")

In [ ]:
def events_to_voxel_grid(events, num_bins, width, height):
    """
    https://github.com/uzh-rpg/rpg_e2vid/blob/master/utils/inference_utils.py
    Build a voxel grid with bilinear interpolation in the time domain from a set of events.

    :param events: a [N x 4] NumPy array (np.int32) containing one event per row in the form:
        [timestamp(us), x, y, polarity(0 or 1)]
    :param num_bins: number of bins in the temporal axis of the voxel grid
    :param width, height: dimensions of the voxel grid
    """

    assert(events.shape[1] == 4)
    assert(num_bins > 0)
    assert(width > 0)
    assert(height > 0)

    voxel_grid = np.zeros((num_bins, height, width), np.float32).ravel()

    # normalize the event timestamps so that they lie between 0 and num_bins
    last_stamp = events[-1, 0]
    first_stamp = events[0, 0]
    deltaT = last_stamp - first_stamp

    if deltaT == 0:
        deltaT = 1.0

    ts = events[:, 0].astype(np.float32)
    ts = (num_bins - 1) * (ts - first_stamp) / deltaT
    xs = events[:, 1].astype(np.int32)
    ys = events[:, 2].astype(np.int32)
    pols = events[:, 3].astype(np.float32)
    pols[pols == 0] = -1  # polarity should be +1 / -1

    tis = ts.astype(np.int32)
    dts = ts - tis
    vals_left = pols * (1.0 - dts)
    vals_right = pols * dts

    valid_indices = tis < num_bins
    np.add.at(voxel_grid, xs[valid_indices] + ys[valid_indices] * width
              + tis[valid_indices] * width * height, vals_left[valid_indices])

    valid_indices = (tis + 1) < num_bins
    np.add.at(voxel_grid, xs[valid_indices] + ys[valid_indices] * width
              + (tis[valid_indices] + 1) * width * height, vals_right[valid_indices])

    voxel_grid = np.reshape(voxel_grid, (num_bins, height, width))

    return voxel_grid

def visual_voxel_grid(voxel_grid, output_folder, filename_key):
    for i in range(voxel_grid.shape[0]):
        path = os.path.join(output_folder, '%s_%02d.png' % (filename_key, i))
        gridmin = np.min(voxel_grid[i])
        gridmax = np.max(voxel_grid[i])
        absmax = max(abs(gridmin), abs(gridmax))
        normalize_im = voxel_grid[i] / absmax * 0.5 + 0.5
        # normalize_im = (voxel_grid[i] - np.min(voxel_grid[i])) / (np.max(voxel_grid[i]) - np.min(voxel_grid[i]))
        cv2.imwrite(path, (normalize_im * 255).astype(np.uint8))

def read_event_dir(event_dir):
    event_dir = Path(event_dir)
    with open(event_dir / 'info.txt', 'r') as f:
        video_info = f.readlines()
    file_timestamps_us = [int(info_i.split()[1]) for info_i in video_info]
    file_paths = [Path(info_i.split()[0]) for info_i in video_info]
    imgs = []
    for i, filepath in enumerate(file_paths):
        img = cv2.imread(event_dir / filepath.name, cv2.IMREAD_GRAYSCALE)
        imgs.append(img)
    return np.array(imgs), np.array(file_timestamps_us) / 1e6  # convert to seconds

def drop_hot_pixels_events(events_df, dominance_thresh=0.95, min_events=50):
    # Compute per-pixel event counts
    pixel_stats = (
        events_df
        .groupby(["x", "y"])["p"]
        .agg(
            total="count",
            pos=lambda s: (s == 1).sum(),
            neg=lambda s: (s == 0).sum()
        )
    )
    # Polarity dominance ratio
    pixel_stats["dominance"] = pixel_stats[["pos", "neg"]].max(axis=1) / pixel_stats["total"]
    
    hot_pixels = pixel_stats[
        (pixel_stats["dominance"] > dominance_thresh) &
        (pixel_stats["total"] >= min_events)
    ].index  # MultiIndex (x, y)
    
    print(f"Removing {len(hot_pixels)} hot pixels.")
    
    events_df_clean = events_df.set_index(["x", "y"]).drop(hot_pixels).reset_index()
    
    return events_df_clean

def read_event_dir_davis(event_dir):
    event_dir = Path(event_dir)
    img_dir = event_dir / "images"
    with open(event_dir / 'events.txt', 'r') as f:
        events_df = pd.read_csv(f, sep=" ", names=["t", "x", "y", "p"])

    with open(event_dir / 'images.txt', 'r') as f:
        video_info = f.readlines()
    file_timestamps_s = [float(info_i.split()[0]) for info_i in video_info]
    file_paths = [Path(info_i.split()[1]) for info_i in video_info]
    imgs = []
    for i, filepath in enumerate(file_paths):
        img = cv2.imread(img_dir / filepath.name, cv2.IMREAD_GRAYSCALE)
        imgs.append(img)
    return events_df, np.array(imgs), np.array(file_timestamps_s)

def make_f_Q_placeholder(
    w_min: float = 0.09,
    scale: float = 7e5,
    kind: str = "beta",          # "triangle", "beta", or "cosine"
    beta_a: float = 2.0,         # used when kind="beta"
    beta_b: float = 2.0,         # used when kind="beta"
) -> np.ndarray:
    """
    Placeholder f_Q lookup table (length 256) for 8-bit APS intensities.

    Returns
    -------
    f_Q : (256,) float64
        Larger values => higher confidence (larger R_bar_inv).
        Intended to be indexed by intensity bin in [0,255] (or [1..256] if you do +1 like MATLAB).
    """
    x = np.linspace(0.0, 1.0, 256, dtype=np.float64)

    if kind == "triangle":
        # Peak at mid-gray, downweight extremes
        w = 1.0 - 2.0 * np.abs(x - 0.5)
        w = np.clip(w, 0.0, 1.0)

    elif kind == "cosine":
        # Smoothly goes to ~0 at extremes, peak at 0.5
        w = np.sin(np.pi * x) ** 2

    elif kind == "beta":
        # Beta(a,b) shape without scipy: proportional to x^(a-1) (1-x)^(b-1)
        # Peaks away from extremes for a,b>1; symmetric when a=b.
        a, b = float(beta_a), float(beta_b)
        w = np.power(np.clip(x, 1e-12, 1.0 - 1e-12), a - 1.0) * np.power(
            np.clip(1.0 - x, 1e-12, 1.0 - 1e-12), b - 1.0
        )
        w /= w.max()

    else:
        raise ValueError("kind must be one of: 'triangle', 'cosine', 'beta'")

    # Clamp like their MATLAB comment (max(weight_func, 0.09) * 7e5)
    w = np.maximum(w, float(w_min))

    # Scale to set overall frame confidence relative to events
    f_Q = w * float(scale)
    return f_Q

# initial data loading (each cell is diff data type)

In [ ]:
import aedat

dataname = "spot_dynamic"
decoder = aedat.Decoder(config.DATA_DIR / f"event/{dataname}.aedat4")
print(decoder.id_to_stream())

events_df = pd.DataFrame(columns=["t", "x", "y", "p"])
timestamps = []
frames = []
exposures = []
event_ts = []
event_xs = []
event_ys = []
event_ps = []
xd = None
for packet in tqdm(decoder):
    if "frame" in packet:
        timestamps.append(packet["frame"]["t"])
        frames.append(packet["frame"]["pixels"])
        exposures.append(packet["frame"]["exposure_end_t"] - packet["frame"]["exposure_begin_t"])
    if "events" in packet:
        event_ts.append(packet["events"]["t"])
        event_xs.append(packet["events"]["x"])
        event_ys.append(packet["events"]["y"])
        event_ps.append(packet["events"]["p"])
event_ts = np.concatenate(event_ts)
event_xs = np.concatenate(event_xs)
event_ys = np.concatenate(event_ys)
event_ps = np.concatenate(event_ps)
events_df = pd.DataFrame({
    "t": event_ts,
    "x": event_xs,
    "y": event_ys,
    "p": event_ps
})
frames = np.array(frames)
timestamps = np.array(timestamps)
exposures = np.array(exposures)
T, H, W = frames.shape

t0_us = events_df["t"].min()
events_df["t"] = (events_df["t"] - t0_us) / 1e6
timestamps = (timestamps - t0_us) / 1e6
exposures = exposures / 1e6

tstart, tend = 0.0, events_df["t"].max()
# shooting-rapid-eccv-2
# tstart, tend = 1.5, 5.0
# shooting-rapid-eccv
# tstart, tend = 4.0, 7.5
# hand_wave
# tstart, tend = 10.0, 15.0
events_df = events_df[(events_df["t"] >= tstart) & (events_df["t"] <= tend)]
aps_timestamp_mask = (timestamps >= tstart) & (timestamps <= tend)
timestamps = timestamps[aps_timestamp_mask]
frames = frames[aps_timestamp_mask]
texp = tend - tstart
events_df = drop_hot_pixels_events(events_df, dominance_thresh=0.95)
events_df.head()

In [ ]:
# converting david 346 aedat4 to mat for AKF paper
ct_mat = (0.143 + 0.225) / 2
events_mat = events_df[["t", "x", "y", "p"]].to_numpy().astype("float64")
events_mat[:, 0] *= 1e6
events_mat[:, 1] += 1
events_mat[:, 2] += 1
exposures_mat = exposures.reshape(1, -1) * 1e6
f_Q = make_f_Q_placeholder(kind="triangle")
frames_mat = frames.transpose(1, 2, 0)
timestamps_mat = timestamps.reshape(-1, 1) * 1e6
mat_outpath = config.DATA_DIR / f"event/{dataname}.mat"
mat_data = {
    "ct": ct_mat,
    "events": events_mat,
    "exposure": exposures_mat,
    "f_Q": f_Q,
    "im_height": frames.shape[1],
    "im_width": frames.shape[2],
    "image": frames_mat,
    "post_process": 1,
    "time_image": timestamps_mat
}
from scipy.io import savemat
savemat(mat_outpath, mat_data)

In [ ]:
from pathlib import Path
import pandas as pd
# DVXplorer csv data
dataname = "teaser6"
path = config.DATA_DIR / Path(f"event/{dataname}.csv")
events_df = pd.read_csv(path, skiprows=1, header=0, names=["t", "x", "y", "p"])
T, H, W = 690, 480, 640
with open(path, "r") as f:
    first = f.readline()
print(first)
# convert times from microseconds to seconds and set starting time to 0
events_df["t"] = (events_df["t"] - events_df["t"].min()) / 1e6
tstart, tend = 0, events_df["t"].max()
events_df.head()
print(f"total capture time: {events_df['t'].max()}")
# balloon1
# tstart, tend = 6.5, 7.5
# seed1
# tstart, tend = 0.0, 7.0
# tstart = 29.0
tend = 7.0
texp = tend - tstart
events_df = events_df[(events_df["t"] >= tstart) & (events_df["t"] <= tend)]
print(f"Total events after filtering: {len(events_df)}")
# drop hot pixels
events_df = drop_hot_pixels_events(events_df, dominance_thresh=0.95)

In [ ]:
dataname = "shapes_6dof"
outdir = config.OUTPUT_DIR / f"results/event/{dataname}"
outdir.mkdir(parents=True, exist_ok=True)
events_df, frames, timestamps = read_event_dir_davis(config.DATA_DIR / f"event/{dataname}")
ihpp_data.to_video(frames, outdir / f"{dataname}-GT.mp4", cmap="gray", playback_fps=30)

tstart, tend = events_df["t"].min(), events_df["t"].max()
# dynamic_6dof
# tstart, tend = 10.0, 15.0
# office_spiral
# tstart, tend = 5.0, 10.0
# shapes_6dof
tstart, tend = 35.0, 40.0
events_df = events_df[(events_df["t"] >= tstart) & (events_df["t"] <= tend)]
events_df = drop_hot_pixels_events(events_df, dominance_thresh=0.95)
aps_timestamp_mask = (timestamps >= tstart) & (timestamps <= tend)
timestamps = timestamps[aps_timestamp_mask]
frames = frames[aps_timestamp_mask]
texp = tend - tstart
T, H, W = frames.shape

# converting david 240 data to mat for AKF paper
ct_mat = 0.11
events_mat = events_df[["t", "x", "y", "p"]].to_numpy().astype("float64")
events_mat[:, 0] *= 1e6
events_mat[:, 1] += 1
events_mat[:, 2] += 1
# guesstimated exposure time
exposures_mat = np.full((1, frames.shape[0]), 10_000, dtype="float64")
f_Q = make_f_Q_placeholder(kind="triangle")
frames_mat = frames.transpose(1, 2, 0)
timestamps_mat = timestamps.reshape(-1, 1) * 1e6
mat_outpath = config.DATA_DIR / f"event/{dataname}.mat"
mat_data = {
    "ct": ct_mat,
    "events": events_mat,
    "exposure": exposures_mat,
    "f_Q": f_Q,
    "im_height": frames.shape[1],
    "im_width": frames.shape[2],
    "image": frames_mat,
    "post_process": 1,
    "time_image": timestamps_mat
}
from scipy.io import savemat
savemat(mat_outpath, mat_data)

In [ ]:
dataname = "moving_flashing_circle"
datapath = Path(f"data/event/{dataname}.txt")
events_df = pd.read_csv(datapath, sep=' ', names=['t', 'x', 'y', 'p'])
tstart, tend = 0.0, 10.0
texp = tend - tstart
T, H, W = 2000, 180, 240

# simulated data, starts at 0
events_df["t"] = (events_df["t"]) / 1e6
events_df

# start the actual run

In [ ]:
# DVS346
k1 = 0.00018 * 29250
k3 = 0.0001
kL = 1.0
c_plus = 0.143
c_minus = 0.225

# DVS240
# k1 = 0.000094 * 47065
# k3 = 0.0002
# kL = 1.0
# c_plus = 0.11
# c_minus = 0.11

# DVXplorer
# i have no idea what these parameters are for this camera
# just use DVS 240 xd

# calculated
A = k1 * kL
mL = (k3 / A) ** 2

ts = events_df["t"].to_numpy()
ts = (ts - tstart) / texp
x = events_df["x"].to_numpy()
y = events_df["y"].to_numpy()
pol = events_df["p"].to_numpy()
pol_delt = np.where(pol > 0, c_plus, -c_minus)

outdir = config.OUTPUT_DIR / Path(f"results/event/{dataname}")
outdir.mkdir(parents=True, exist_ok=True)

In [ ]:
# misc
nt, ny, nx = 1000, H, W
# shooting-rapid-eccv-2, shooting-rapid-eccv
# nt, ny, nx = 3500, H, W
# 5 seconds recording length for 1000 fps
# nt, ny, nx = 5000, H, W
tt = np.linspace(0, 1, nt)
yy = np.linspace(0, 1, ny)
xx = np.linspace(0, 1, nx)
texp = tend - tstart
nmodes = (nt, ny, nx)
V = texp * H * W
F = finufft.nufft3d1(
    ts * np.pi * 2, y / H * np.pi * 2, x / W * np.pi * 2, pol_delt.astype("complex128"), n_modes=nmodes, eps=1e-6, isign=-1
) / np.sqrt(V)

In [ ]:
def soft_thresh_sigmoid(F, bound, steepness=1.0):
    F_abs = np.abs(F)
    # Ratio > 1 means above threshold
    ratio = F_abs / bound
    
    # Sigmoid centered at ratio=1
    weights = 1.0 / (1.0 + np.exp(-steepness * (ratio - 1.0)))
    return weights

In [ ]:
sumc2 = np.sum(pol_delt**2)
# alpha = np.prod(nmodes) // 2
alpha = 0.0001
bound = np.sqrt(chi2.ppf(1 - alpha, 2) * sumc2 / (2 * V))
cfar_mask = np.abs(F) >= bound
print(f"CFAR threshold: {bound:.4f}, kept {np.count_nonzero(cfar_mask)}/{F.size} frequencies")
F_bd = F.copy()

# hard threshold
F_bd[~cfar_mask] = 0.0
# soft threshold something
# gain = soft_thresh_sigmoid(F, bound, steepness=100.0)
# F_bd *= gain

ft = np.fft.fftshift(np.fft.fftfreq(nmodes[0], d=1/nmodes[0])) * 2 * np.pi
fy = np.fft.fftshift(np.fft.fftfreq(nmodes[1], d=1/nmodes[1])) * 2 * np.pi
fx = np.fft.fftshift(np.fft.fftfreq(nmodes[2], d=1/nmodes[2])) * 2 * np.pi
ftgrid, fygrid, fxgrid = np.meshgrid(ft, fy, fx, indexing='ij')
# drop DC drift... is this correct in the KF?
# F_bd[ftgrid == 0] = 0.0
# high pass on 
# F_bd[np.abs(ftgrid) <= 5] = 0.0

# integrator with hard highpass
highpass_thresh = 5.0
inv_i_omega = np.zeros_like(ftgrid, dtype="complex128")
# Create a mask to avoid division by zero at the DC component (freqs == 0)
mask = ftgrid != 0
# Apply the integration rule: 1 / (i * 2 * pi * f)
inv_i_omega[mask] = 1.0 / (2j * np.pi * ftgrid[mask])
inv_i_omega[np.abs(ftgrid) <= highpass_thresh] = 0.0

# leaky integrator 1
# leak = 10.0
# inv_i_omega = 1.0 / (leak + 2j * np.pi * ftgrid)

# some other kind of leaky integrator idk
# omega = 2 * np.pi * ftgrid
# cutoff = 2.0
# omega0 = 2 * np.pi * cutoff
# hp = (omega ** 2) / (omega ** 2 + omega0 ** 2)
# inv_i_omega = np.zeros_like(omega, dtype=np.complex128)
# mask = omega != 0
# inv_i_omega[mask] = (1.0 / (1j*omega[mask])) * hp[mask]

# reg1 = 1
# reg2 = 1
# inv_i_omega = (-1j * ftgrid) / (ftgrid ** 2 + reg1 * (fxgrid ** 2 + fygrid ** 2) + reg2)
# inv_i_omega[ftgrid == 0] = 0.0

F_bd_integrated = F_bd * inv_i_omega

# k = np.sqrt(fxgrid**2 + fygrid**2)
# kc = 2.0  # cutoff in same units as fxgrid/fygrid (tune)
# LP = np.exp(-0.5 * (k / kc)**2)     # spatial low-pass
# HP = 1.0 - LP                       # spatial high-pass
# a = 10  # sharpening strength (tune)
# H_sharp = 1.0 + a * HP                    # unsharp mask gain
# F_sharp = F_bd_integrated * H_sharp

recon_der = np.fft.ifftn(np.fft.ifftshift(F_bd), axes=(0, 1, 2), norm="forward") / np.sqrt(V)
recon_der = recon_der.real
recon_integ = np.fft.ifftn(np.fft.ifftshift(F_bd_integrated), axes=(0, 1, 2), norm="forward") / np.sqrt(V)
recon_integ = recon_integ.real
# recon_integ_exp = np.exp(recon_integ / A)

del inv_i_omega, mask, cfar_mask, ftgrid, fygrid, fxgrid

In [ ]:
recon_der_absmax = np.abs(recon_der).max()
ihpp_data.to_video(recon_der, outdir / f"recon_derivative_eventsonly.mp4", cmap="RdBu_r", playback_fps=30, vmin=-recon_der_absmax, vmax=recon_der_absmax)
# ihpp_data.to_video(recon_integ_sharp, outdir / f"recon_integ_sharp.mp4", cmap="gray", playback_fps=30)
ihpp_data.to_video(recon_integ, outdir / f"recon_integral_eventsonly.mp4", cmap="gray", playback_fps=30)

In [ ]:
# dynamic_6dof
# mult = 20
# office_spiral
# mult = 20
# shapes_6dof
# mult = 30
# misc
mult = 20

asdf_exp = np.exp(recon_integ*mult)
ihpp_data.to_video(asdf_exp, outdir / f"recon_integral_exp_eventsonly.mp4", cmap="gray", playback_fps=30)
from skimage import exposure
p2, p98 = np.percentile(asdf_exp, (2, 98))
img_rescale = exposure.rescale_intensity(asdf_exp, in_range=(p2, p98))
ihpp_data.to_video(img_rescale, outdir / f"recon_integral_exp_rescaled_eventsonly.mp4", cmap="gray", playback_fps=30)
# ihpp_data.to_video(img_rescale, outdir / f"recon_integral_exp_rescaled_eventsonly", cmap="gray")

In [ ]:
events_arr = np.array(events_df[['t', 'x', 'y', 'p']])
events_voxel = events_to_voxel_grid(events_arr, num_bins=nt, width=W, height=H)
abshighest = np.max(np.abs(events_voxel))
# abshighest = 1.0
ihpp_data.to_video(events_voxel, outdir / "events_raw.mp4", cmap="RdBu_r", playback_fps=30, vmin=-abshighest, vmax=abshighest)
ihpp_data.to_video(events_voxel, outdir / "events_raw", cmap="RdBu_r", vmin=-abshighest, vmax=abshighest)
events_integ_step = np.cumsum(events_voxel, axis=0)
ihpp_data.to_video(events_integ_step, outdir / f"step_integ.mp4", playback_fps=30, cmap="gray")
# ihpp_data.to_video(np.exp(events_integ_step / A), outdir / f"step_integ_exp.mp4", playback_fps=30, cmap="gray")

In [ ]:
events_integ_step = np.cumsum(events_voxel[6800:], axis=0)
ihpp_data.to_video(events_integ_step, outdir / f"step_integ.mp4", playback_fps=30, cmap="gray", use_quantile=True)

---

# kalman filtering part 2 electric boogaloo

vibe coding

In [ ]:
"""
Kalman Filter Fusion — Drift-Corrected Version
================================================

Key fixes over v1:
1. CONTINUOUS MEASUREMENT INTERPOLATION: between APS frames, we interpolate
   both the log-intensity measurement and its covariance, applying a correction
   pull at *every* event timestamp (not just at APS arrivals).  This mirrors
   the continuous Kalman-Bucy formulation in the AKF paper (Wang et al.),
   Eq. (14)-(17).

2. FIXED PROCESS NOISE: removed the adaptive Q that scaled by 1/λ(t),
   which created a positive-feedback loop where upward drift → small Q →
   small P → small K → drift ignored.  Process noise is now constant or
   optionally scaled by the *measurement-derived* intensity (not the state).

3. SPATIALLY-VARYING SCALE: estimate per-pixel (or per-patch) scale factor
   to prevent DC bias accumulation from a single global scalar.

4. OPTIONAL BIAS STATE: augment the scalar KF with a slowly-varying bias
   b_p(t) that captures systematic drift in the event derivative.

Two filters:
  - SpatialDomainKF_v2   (per-pixel, log-intensity)
  - FourierDomainKF_v2   (per-frequency-bin, 2D DFT)
"""

import numpy as np
from typing import Optional, Tuple


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def intensity_to_log(frame: np.ndarray, I0: float = 1.0) -> np.ndarray:
    return np.log(np.clip(frame, 0, None) + I0)


def log_domain_measurement_variance(
    frame: np.ndarray,
    sigma2_im: float = 0.1,
    I0: float = 1.0,
    R_max: float = 10.0,
) -> np.ndarray:
    intensity = np.clip(frame, 0, None).astype(np.float64)
    I_max = intensity.max() + 1e-8
    f_w = 4.0 * (intensity / I_max) * (1.0 - intensity / I_max)
    f_w = np.clip(f_w, 1e-4, 1.0)
    R_bar = sigma2_im / f_w
    R_bar = np.clip(R_bar, 0, R_max)
    R_log = R_bar / (intensity + I0) ** 2
    return np.clip(R_log, 1e-8, R_max)


# ===================================================================
# 1.  SPATIAL-DOMAIN KF v2 — with continuous measurement pull
# ===================================================================
class SpatialDomainKF_v2:
    """
    Per-pixel Kalman filter in log-intensity space, with continuous
    inter-frame correction.

    State model (following AKF):
        dL_p = e_p(t) dt + dw_p          (prediction from events + process noise)
        z_p(t) = L_p(t) + μ_p            (APS measurement = truth + noise)

    Between APS frames k and k+1, we linearly interpolate the measurement:
        L^A_p(t) = lerp(L^A_p(τ_k), L^A_p(τ_{k+1}), (t-τ_k)/(τ_{k+1}-τ_k))
        R_p(t)   = lerp(R_p(τ_k), R_p(τ_{k+1}), ...)

    The filter update at each event timestamp becomes (discretized AKF Eq.17):
        K_p(t) = P_p(t) / (P_p(t) + R_p(t))
        x_p(t) = x_p(t) + K_p(t) * (L^A_p(t) - x_p(t))  [correction pull]

    This ensures the state is always being pulled toward the interpolated
    frame, preventing open-loop drift between APS arrivals.

    Optional bias-state augmentation:
        If estimate_bias=True, we track a per-pixel bias b_p that evolves as
        a random walk:  b_p(t+dt) = b_p(t) + noise
        The prediction becomes:  x_p(t+dt) = x_p(t) + (e_p - b_p) * dt
        and b_p is updated whenever an APS frame provides ground truth.
    """

    def __init__(
        self,
        H: int,
        W: int,
        process_var: float = 0.01,
        sigma2_im: float = 0.1,
        I0: float = 1.0,
        init_var: float = 10.0,
        estimate_bias: bool = True,
        bias_process_var: float = 1e-4,
    ):
        self.H, self.W = H, W
        self.process_var = process_var
        self.sigma2_im = sigma2_im
        self.I0 = I0
        self.estimate_bias = estimate_bias
        self.bias_process_var = bias_process_var

        # State
        self.x = np.zeros((H, W), dtype=np.float64)         # log-intensity
        self.P = np.full((H, W), init_var, dtype=np.float64) # state variance

        if estimate_bias:
            self.b = np.zeros((H, W), dtype=np.float64)      # bias estimate
            self.Pb = np.full((H, W), 0.1, dtype=np.float64) # bias variance

        self.initialized = False

        # Interpolated measurement cache
        # We store the two bracketing APS frames for interpolation
        self._z_prev = None       # log(APS_k + I0)
        self._z_next = None       # log(APS_{k+1} + I0)
        self._R_prev = None       # R_p(τ_k)
        self._R_next = None       # R_p(τ_{k+1})
        self._t_prev = None       # τ_k
        self._t_next = None       # τ_{k+1}

    def _interp_measurement(self, t: float) -> Tuple[np.ndarray, np.ndarray]:
        """Linearly interpolate the APS measurement and its covariance at time t.

        If t is outside the bracket, we clamp to the nearest frame (no
        extrapolation — that would be inventing data).
        """
        if self._z_prev is None:
            # No frames yet — return high-uncertainty dummy
            return self.x.copy(), np.full((self.H, self.W), 1e6)

        if self._z_next is None:
            # Only one frame so far — use it as-is
            return self._z_prev.copy(), self._R_prev.copy()

        # Linear interpolation weight
        dt_total = self._t_next - self._t_prev
        if dt_total < 1e-12:
            alpha = 0.5
        else:
            alpha = np.clip((t - self._t_prev) / dt_total, 0.0, 1.0)

        z_interp = (1 - alpha) * self._z_prev + alpha * self._z_next
        R_interp = (1 - alpha) * self._R_prev + alpha * self._R_next
        return z_interp, R_interp

    def set_aps_bracket(
        self,
        z_prev: np.ndarray, R_prev: np.ndarray, t_prev: float,
        z_next: Optional[np.ndarray], R_next: Optional[np.ndarray],
        t_next: Optional[float],
    ):
        """Set the pair of APS frames used for interpolation."""
        self._z_prev = z_prev
        self._R_prev = R_prev
        self._t_prev = t_prev
        self._z_next = z_next
        self._R_next = R_next
        self._t_next = t_next

    def step(self, derivative: np.ndarray, dt: float, t: float):
        """Combined predict + interpolated-measurement correction.

        This replaces the separate predict/update calls.  At every event
        timestamp we:
          1. Predict forward using the event derivative
          2. Pull toward the interpolated APS measurement

        Parameters
        ----------
        derivative : (H, W)  scaled d/dt[log λ]
        dt         : time since last step
        t          : current absolute time (for interpolation)
        """
        if not self.initialized:
            return

        # ----- Prediction --------------------------------------------------
        if self.estimate_bias:
            corrected_deriv = derivative - self.b
        else:
            corrected_deriv = derivative

        self.x = self.x + corrected_deriv * dt
        Q = self.process_var * dt
        self.P = self.P + Q

        if self.estimate_bias:
            self.Pb = self.Pb + self.bias_process_var * dt

        # ----- Continuous measurement correction ---------------------------
        z_interp, R_interp = self._interp_measurement(t)
        S = self.P + R_interp
        K = self.P / S                       # (H, W) Kalman gain
        innovation = z_interp - self.x
        self.x = self.x + K * innovation
        self.P = (1.0 - K) * self.P

    def update_aps(self, aps_frame: np.ndarray):
        """Hard measurement update when an APS frame actually arrives.

        This is applied in addition to the continuous correction, to get
        the exact (non-interpolated) frame data.  It also updates the
        bias state if enabled.
        """
        z = intensity_to_log(aps_frame, self.I0)
        R = log_domain_measurement_variance(aps_frame, self.sigma2_im, self.I0)

        if not self.initialized:
            self.x = z.copy()
            self.P = R.copy()
            self.initialized = True
            return

        # Standard Kalman update
        S = self.P + R
        K = self.P / S
        innovation = z - self.x
        self.x = self.x + K * innovation
        self.P = (1.0 - K) * self.P

        # Bias update: the innovation at an APS frame tells us
        # how much the predictions have been systematically off
        if self.estimate_bias:
            # The bias should absorb the fraction of innovation
            # attributable to systematic derivative error
            Sb = self.Pb + self.P + R
            Kb = self.Pb / Sb
            # Positive innovation (measurement > prediction) means the
            # derivative was biased low → increase b to subtract less
            # ... actually, b is subtracted from the derivative, so if
            # we've been overshooting (x > z), innovation < 0, we want
            # b to increase (subtract more from derivative next time).
            self.b = self.b - Kb * innovation
            self.Pb = (1.0 - Kb) * self.Pb

    def get_state(self) -> np.ndarray:
        return self.x.copy()

    def get_bias(self) -> np.ndarray:
        if self.estimate_bias:
            return self.b.copy()
        return np.zeros((self.H, self.W))

    def get_intensity(self) -> np.ndarray:
        return np.exp(self.x) - self.I0


def run_spatial_kalman_v2(
    event_derivatives: np.ndarray,   # (T_event, H, W)
    event_timestamps: np.ndarray,    # (T_event,)
    aps_frames: np.ndarray,          # (T_aps, H, W)
    aps_timestamps: np.ndarray,      # (T_aps,)
    scale_factor: float = 1.0,
    process_var: float = 0.01,
    sigma2_im: float = 0.1,
    I0: float = 1.0,
    init_var: float = 10.0,
    estimate_bias: bool = True,
    bias_process_var: float = 1e-4,
) -> np.ndarray:
    """
    Full pipeline with continuous inter-frame correction.

    Returns
    -------
    reconstructed : (T_event, H, W)  log-intensity at every event timestamp
    """
    T_event, H, W = event_derivatives.shape
    T_aps = aps_frames.shape[0]

    kf = SpatialDomainKF_v2(
        H, W, process_var, sigma2_im, I0, init_var,
        estimate_bias, bias_process_var,
    )

    # Precompute all APS log-frames and covariances
    aps_log = np.stack([intensity_to_log(f, I0) for f in aps_frames])
    aps_R = np.stack([
        log_domain_measurement_variance(f, sigma2_im, I0) for f in aps_frames
    ])

    # Initialise from first APS frame at or before event stream
    init_idx = np.searchsorted(aps_timestamps, event_timestamps[0], side="right") - 1
    if init_idx < 0:
        init_idx = 0
    kf.update_aps(aps_frames[init_idx])

    # Set up interpolation brackets
    # aps_bracket_idx tracks which pair of APS frames we're between
    aps_bracket_idx = init_idx  # we're between aps[bracket_idx] and aps[bracket_idx+1]

    def update_bracket(bracket_idx):
        if bracket_idx + 1 < T_aps:
            kf.set_aps_bracket(
                aps_log[bracket_idx], aps_R[bracket_idx], aps_timestamps[bracket_idx],
                aps_log[bracket_idx + 1], aps_R[bracket_idx + 1], aps_timestamps[bracket_idx + 1],
            )
        else:
            # Past the last frame — use last frame with no forward bracket
            kf.set_aps_bracket(
                aps_log[bracket_idx], aps_R[bracket_idx], aps_timestamps[bracket_idx],
                None, None, None,
            )

    update_bracket(aps_bracket_idx)

    output = np.zeros((T_event, H, W), dtype=np.float64)
    prev_t = event_timestamps[0]
    next_aps_idx = init_idx + 1  # next APS frame to process

    for i in range(T_event):
        t = event_timestamps[i]
        dt = t - prev_t if i > 0 else 0.0

        # --- Check if any APS frames have arrived --------------------------
        while next_aps_idx < T_aps and aps_timestamps[next_aps_idx] <= t:
            kf.update_aps(aps_frames[next_aps_idx])
            aps_bracket_idx = next_aps_idx
            update_bracket(aps_bracket_idx)
            next_aps_idx += 1

        # --- Predict + continuous correction -------------------------------
        if dt > 0:
            kf.step(event_derivatives[i] * scale_factor, dt, t)

        output[i] = kf.get_state()
        prev_t = t

    return output


# ===================================================================
# 2.  FOURIER-DOMAIN KF v2 — with continuous measurement pull
# ===================================================================
class FourierDomainKF_v2:
    """
    Per-frequency Kalman filter with continuous inter-frame correction
    and CFAR-informed frequency weighting.

    Same continuous-correction philosophy as the spatial filter, but
    operating on 2D DFT coefficients.
    """

    def __init__(
        self,
        H: int,
        W: int,
        process_var: float = 0.01,
        sigma2_im: float = 0.1,
        I0: float = 1.0,
        init_var: float = 10.0,
        freq_weighting: Optional[np.ndarray] = None,
        estimate_bias: bool = True,
        bias_process_var: float = 1e-4,
    ):
        self.H, self.W = H, W
        self.process_var = process_var
        self.sigma2_im = sigma2_im
        self.I0 = I0
        self.estimate_bias = estimate_bias
        self.bias_process_var = bias_process_var

        self.X = np.zeros((H, W), dtype=np.complex128)
        self.P = np.full((H, W), init_var, dtype=np.float64)

        if estimate_bias:
            self.B = np.zeros((H, W), dtype=np.complex128)  # Fourier bias
            self.Pb = np.full((H, W), 0.1, dtype=np.float64)

        self.initialized = False

        if freq_weighting is not None:
            self.freq_weight = np.clip(freq_weighting.astype(np.float64), 1e-3, 1.0)
        else:
            self.freq_weight = np.ones((H, W), dtype=np.float64)

        # Interpolation cache (in Fourier domain)
        self._Z_prev = None
        self._Z_next = None
        self._Rf_prev = None
        self._Rf_next = None
        self._t_prev = None
        self._t_next = None

    def _measurement_noise_fourier(self, aps_frame: np.ndarray) -> np.ndarray:
        R_spatial = log_domain_measurement_variance(aps_frame, self.sigma2_im, self.I0)
        R_f = np.abs(np.fft.fft2(R_spatial)) / (self.H * self.W)
        return np.clip(R_f, 1e-8, None)

    def _interp_measurement(self, t: float):
        if self._Z_prev is None:
            return self.X.copy(), np.full((self.H, self.W), 1e6)
        if self._Z_next is None:
            return self._Z_prev.copy(), self._Rf_prev.copy()

        dt_total = self._t_next - self._t_prev
        if dt_total < 1e-12:
            alpha = 0.5
        else:
            alpha = np.clip((t - self._t_prev) / dt_total, 0.0, 1.0)

        Z_interp = (1 - alpha) * self._Z_prev + alpha * self._Z_next
        Rf_interp = (1 - alpha) * self._Rf_prev + alpha * self._Rf_next
        return Z_interp, Rf_interp

    def set_aps_bracket(
        self,
        Z_prev, Rf_prev, t_prev,
        Z_next, Rf_next, t_next,
    ):
        self._Z_prev = Z_prev
        self._Rf_prev = Rf_prev
        self._t_prev = t_prev
        self._Z_next = Z_next
        self._Rf_next = Rf_next
        self._t_next = t_next

    def step(self, derivative: np.ndarray, dt: float, t: float):
        if not self.initialized:
            return

        dX = np.fft.fft2(derivative) * dt

        if self.estimate_bias:
            dX = dX - self.B * dt

        self.X = self.X + dX
        # Process noise: inflate for low-CFAR-confidence frequencies
        Q_f = self.process_var * dt / self.freq_weight
        self.P = self.P + Q_f

        if self.estimate_bias:
            self.Pb = self.Pb + self.bias_process_var * dt

        # Continuous correction
        Z_interp, Rf_interp = self._interp_measurement(t)
        S = self.P + Rf_interp
        K = self.P / S
        innovation = Z_interp - self.X
        self.X = self.X + K * innovation
        self.P = (1.0 - K) * self.P

    def update_aps(self, aps_frame: np.ndarray):
        z_spatial = intensity_to_log(aps_frame, self.I0)
        Z = np.fft.fft2(z_spatial)
        R_f = self._measurement_noise_fourier(aps_frame)

        if not self.initialized:
            self.X = Z.copy()
            self.P = R_f.copy()
            self.initialized = True
            return

        S = self.P + R_f
        K = self.P / S
        innovation = Z - self.X
        self.X = self.X + K * innovation
        self.P = (1.0 - K) * self.P

        if self.estimate_bias:
            Sb = self.Pb + self.P + R_f
            Kb = self.Pb / Sb
            self.B = self.B - Kb * innovation
            self.Pb = (1.0 - Kb) * self.Pb

    def get_state_spatial(self) -> np.ndarray:
        return np.real(np.fft.ifft2(self.X))

    def get_intensity(self) -> np.ndarray:
        return np.exp(self.get_state_spatial()) - self.I0


def run_fourier_kalman_v2(
    event_derivatives: np.ndarray,
    event_timestamps: np.ndarray,
    aps_frames: np.ndarray,
    aps_timestamps: np.ndarray,
    scale_factor: float = 1.0,
    process_var: float = 0.01,
    sigma2_im: float = 0.1,
    I0: float = 1.0,
    init_var: float = 10.0,
    freq_weighting: Optional[np.ndarray] = None,
    estimate_bias: bool = True,
    bias_process_var: float = 1e-4,
) -> np.ndarray:
    T_event, H, W = event_derivatives.shape
    T_aps = aps_frames.shape[0]

    kf = FourierDomainKF_v2(
        H, W, process_var, sigma2_im, I0, init_var,
        freq_weighting, estimate_bias, bias_process_var,
    )

    # Precompute Fourier-domain APS data
    aps_Z = np.stack([np.fft.fft2(intensity_to_log(f, I0)) for f in aps_frames])
    aps_Rf = np.stack([kf._measurement_noise_fourier(f) for f in aps_frames])

    init_idx = max(0, np.searchsorted(aps_timestamps, event_timestamps[0], side="right") - 1)
    kf.update_aps(aps_frames[init_idx])

    aps_bracket_idx = init_idx

    def update_bracket(idx):
        if idx + 1 < T_aps:
            kf.set_aps_bracket(
                aps_Z[idx], aps_Rf[idx], aps_timestamps[idx],
                aps_Z[idx + 1], aps_Rf[idx + 1], aps_timestamps[idx + 1],
            )
        else:
            kf.set_aps_bracket(
                aps_Z[idx], aps_Rf[idx], aps_timestamps[idx],
                None, None, None,
            )

    update_bracket(aps_bracket_idx)

    output = np.zeros((T_event, H, W), dtype=np.float64)
    prev_t = event_timestamps[0]
    next_aps_idx = init_idx + 1

    for i in range(T_event):
        t = event_timestamps[i]
        dt = t - prev_t if i > 0 else 0.0

        while next_aps_idx < T_aps and aps_timestamps[next_aps_idx] <= t:
            kf.update_aps(aps_frames[next_aps_idx])
            aps_bracket_idx = next_aps_idx
            update_bracket(aps_bracket_idx)
            next_aps_idx += 1

        if dt > 0:
            kf.step(event_derivatives[i] * scale_factor, dt, t)

        output[i] = kf.get_state_spatial()
        prev_t = t

    return output


# ===================================================================
# 3.  SPATIALLY-VARYING SCALE ESTIMATION
# ===================================================================
def estimate_scale_factor_spatial(
    event_derivatives: np.ndarray,
    event_timestamps: np.ndarray,
    aps_frames: np.ndarray,
    aps_timestamps: np.ndarray,
    I0: float = 1.0,
    patch_size: int = 0,
) -> np.ndarray:
    """Per-pixel (or per-patch) scale factor estimation.

    α_p = <∫ė_p dt , Δ log I_p> / <∫ė_p dt , ∫ė_p dt>

    Parameters
    ----------
    patch_size : if > 0, compute per-patch (patch_size x patch_size) and
                 upsample.  0 = per-pixel.  Recommended: 8 or 16 for
                 robustness against noise.

    Returns
    -------
    scale : (H, W) per-pixel scale factors
    """
    _, H, W = event_derivatives.shape
    T_aps = aps_frames.shape[0]
    log_frames = intensity_to_log(aps_frames, I0)

    numerator = np.zeros((H, W), dtype=np.float64)
    denominator = np.zeros((H, W), dtype=np.float64)

    for k in range(T_aps - 1):
        t0, t1 = aps_timestamps[k], aps_timestamps[k + 1]
        mask = (event_timestamps > t0) & (event_timestamps <= t1)
        if mask.sum() < 2:
            continue
        idx = np.where(mask)[0]
        ts = event_timestamps[idx]
        derivs = event_derivatives[idx]
        dt = np.diff(ts)
        integrated = np.zeros((H, W))
        for j in range(len(dt)):
            integrated += 0.5 * (derivs[j] + derivs[j + 1]) * dt[j]

        delta_log = log_frames[k + 1] - log_frames[k]
        numerator += integrated * delta_log
        denominator += integrated * integrated

    scale = np.where(denominator > 1e-10, numerator / denominator, 1.0)

    # Smooth if using patches
    if patch_size > 0:
        from scipy.ndimage import uniform_filter
        scale = uniform_filter(scale, size=patch_size, mode='reflect')

    # Clamp to reasonable range
    median_scale = np.median(scale[np.isfinite(scale)])
    scale = np.clip(scale, 0.1 * median_scale, 10.0 * median_scale)

    return scale


def estimate_scale_factor_global(
    event_derivatives: np.ndarray,
    event_timestamps: np.ndarray,
    aps_frames: np.ndarray,
    aps_timestamps: np.ndarray,
    I0: float = 1.0,
) -> float:
    """Global scalar scale factor (same as v1)."""
    scale_map = estimate_scale_factor_spatial(
        event_derivatives, event_timestamps, aps_frames, aps_timestamps, I0, patch_size=0
    )
    # Robust global estimate: median of per-pixel scales
    return float(np.median(scale_map[np.isfinite(scale_map)]))

In [ ]:
t_event = tt * texp + tstart
t_aps = timestamps

event_derivs = recon_der

# Generate noisy APS frames (intensity domain)
aps = frames

# --- Calibrate scale ---------------------------------------------------
alpha = estimate_scale_factor_global(event_derivs, t_event, aps, t_aps, I0=1.0)
print(f"Estimated scale factor: {alpha:.4f}")

# --- Run both filters --------------------------------------------------
print("Running spatial-domain Kalman filter ...")
# dynamic_6dof
process_var = 1e-3
sigma2_im = 1e-0
# office_spiral (use fourier KF)
# process_var = 1e-3
# sigma2_im = 1e-1
# shapes_6dof (use spatial KF)
# process_var = 1e-4
# sigma2_im = 0.5
recon_spatial = run_spatial_kalman_v2(
    event_derivs, t_event, aps, t_aps,
    scale_factor=alpha, process_var=process_var, sigma2_im=sigma2_im,
    estimate_bias=True, bias_process_var=1e-4,
)

print("Running Fourier-domain Kalman filter ...")
recon_fourier = run_fourier_kalman_v2(
    event_derivs, t_event, aps, t_aps,
    scale_factor=alpha, process_var=process_var, sigma2_im=sigma2_im,
    estimate_bias=True, bias_process_var=1e-4,
)
ihpp_data.to_video(recon_spatial, outdir / f"recon_spatial_kalman.mp4", cmap="gray", playback_fps=30)
ihpp_data.to_video(recon_fourier, outdir / f"recon_fourier_kalman.mp4", cmap="gray", playback_fps=30)

In [ ]:
frames.shape

In [ ]:
plt.imshow(frames[135])

In [ ]:
# For each APS timestamp, find the index in t_event that is closest in time
aps_to_event_idx = np.searchsorted(t_event, t_aps)
# Correct for edge cases where the closest is actually the previous index
aps_to_event_idx = np.clip(aps_to_event_idx, 0, len(t_event) - 1)
# Optionally, check if previous index is closer
for i, t_aps_i in enumerate(t_aps):
    idx = aps_to_event_idx[i]
    if idx > 0 and abs(t_event[idx-1] - t_aps_i) < abs(t_event[idx] - t_aps_i):
        aps_to_event_idx[i] = idx - 1

# aps_to_event_idx now contains the indices in t_event closest to each t_aps
print(aps_to_event_idx)

In [ ]:
aps_framenames = [f"frame_{i:05d}" for i in aps_to_event_idx]
ihpp_data.to_video(aps, outdir / f"aps_frames", cmap="gray", framenames=aps_framenames)

In [ ]:
# dynamic_6dof
mult = 1/5
# shapes_6dof
# mult = 1/10
# office_spiral
# mult = 1/5

In [ ]:
# dynamic_6dof
mult = 1/5
ihpp_data.to_video(np.exp(recon_spatial*mult), outdir / f"recon_spatial_kalman_exp.mp4", cmap="gray", playback_fps=30, vmin=0)
ihpp_data.to_video(np.exp(recon_spatial*mult), outdir / f"recon_spatial_kalman_exp", cmap="gray", vmin=0)

# shapes_6dof
# mult = 1/10
# ihpp_data.to_video(np.exp(recon_spatial*mult), outdir / f"recon_spatial_kalman_exp.mp4", cmap="gray", playback_fps=30, vmin=0)
# ihpp_data.to_video(np.exp(recon_spatial*mult), outdir / f"recon_spatial_kalman_exp", cmap="gray", vmin=0)

# office_spiral
# mult = 1/5
# ihpp_data.to_video(np.exp(recon_fourier*mult), outdir / f"recon_fourier_kalman_exp.mp4", cmap="gray", playback_fps=30, vmin=0)
# ihpp_data.to_video(np.exp(recon_fourier*mult), outdir / f"recon_fourier_kalman_exp", cmap="gray", vmin=0)

In [ ]:
# fourier domain
from skimage import exposure
asdfexp = np.exp(recon_fourier*mult)
p2, p98 = np.percentile(asdfexp, (2, 98))
img_rescale = exposure.rescale_intensity(asdfexp, in_range=(p2, p98))
ihpp_data.to_video(img_rescale, outdir / f"recon_fourier_kalman_rescale.mp4", cmap="gray", playback_fps=30, vmin=0)
ihpp_data.to_video(img_rescale, outdir / f"recon_fourier_kalman_rescale", cmap="gray", vmin=0)

In [ ]:
# spatial domain
asdfexp = np.exp(recon_spatial*mult)
from skimage import exposure
p2, p98 = np.percentile(asdfexp, (2, 98))
img_rescale = exposure.rescale_intensity(asdfexp, in_range=(p2, p98))
ihpp_data.to_video(img_rescale, outdir / f"recon_spatial_kalman_rescale.mp4", cmap="gray", playback_fps=30, vmin=0)
ihpp_data.to_video(img_rescale, outdir / f"recon_spatial_kalman_rescale", cmap="gray", playback_fps=30, vmin=0)

In [ ]:
from skimage import exposure
img_eq = exposure.equalize_hist(asdfexp)
# plt.hist(asdfexp.flatten() / asdfexp.max(), bins=100)

In [ ]:
img_adapteq = exposure.equalize_adapthist(asdfexp / asdfexp.max(), clip_limit=0.03)

In [ ]:
p2, p98 = np.percentile(asdfexp, (2, 98))
img_rescale = exposure.rescale_intensity(asdfexp, in_range=(p2, p98))

In [ ]:
ihpp_data.to_video(img_eq, outdir / f"recon_spatial_kalman_eq.mp4", cmap="gray", playback_fps=30, vmin=0)

In [ ]:
ihpp_data.to_video(img_adapteq, outdir / f"recon_spatial_kalman_adapteq.mp4", cmap="gray", playback_fps=30, vmin=0)

In [ ]:
ihpp_data.to_video(img_rescale, outdir / f"recon_spatial_kalman_rescale.mp4", cmap="gray", playback_fps=30, vmin=0)

In [ ]:
process_var = 1e-3
sigma2_im = 1e-1
recon_spatial = run_spatial_kalman_v2(
    event_derivs, t_event, aps, t_aps,
    scale_factor=alpha, process_var=process_var, sigma2_im=sigma2_im,
    estimate_bias=True, bias_process_var=1e-4,
)

In [ ]:
ihpp_data.to_video(np.exp(recon_spatial/10), outdir / f"recon_spatial_kf_exp.mp4", cmap="gray", playback_fps=30, vmin=0)

In [ ]:
ihpp_data.to_video(np.exp(recon_fourier / 5), outdir / f"recon_fourier_kf_exp.mp4", cmap="gray", playback_fps=30)

In [ ]:
plt.imshow(recon_spatial[20])